# 2. Drifter data cleaning

**What it does**
- Opens the global Clouddrift `gdp1h` hourly arrays (about 197 million readings) without loading them into memory.
- Reads them 2 million readings at a time and keeps those in the study box (94°W–65°W, 54°S–4°S) and period (2001–2022).
- Converts SST from Kelvin to °C and drops non-finite values and anything outside −2 to 35 °C.
- Recovers each reading's drifter ID from `rowsize`, and saves the result once as a small regional extract. Later runs load the extract directly.

**Inputs**
- `data/cache/{lon,lat,time,sst,id,rowsize}.npy` (Clouddrift `gdp1h`, first run only)

**Outputs**
- `data/cache/forecast_obs.npz`: about 4.1 million readings with id, time, lon, lat and SST in °C
- `df`, one row per reading (shown at the end)

Run this notebook from the `notebooks/` folder, top to bottom. The first run takes a few minutes; afterwards it takes seconds and peaks at about 1.5 GB of memory.

## Load and filter the drifter readings

This uses the same filter as her script: 94°W–65°W, 54°S–4°S, 2001–2022, and SST between −2 and 35 °C. The difference is that it reads the global arrays from disk in chunks and saves only the matching readings to `data/cache/forecast_obs.npz`. Her version loads all of them at once (more than 10 GB). The extract is built once, and later runs load the small file.

Not copied: her `valid_velocity.npy` and `valid_sst_velocity.npy` masks. Nothing in this notebook uses them.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

CACHE = Path("../data/cache")
EXTRACT = CACHE / "forecast_obs.npz"
pd.set_option("display.width", 200)

# Study time - 2001 to 2022
START, END = np.datetime64("2001-01-01", "ns"), np.datetime64("2023-01-01", "ns")

# Selected region - Humboldt Region
R = dict(lon=(-94, -65), lat=(-54, -4))


def download_gdp1h(variables):
    """Download any missing Clouddrift gdp1h arrays into data/cache, one variable at a time."""
    missing = [v for v in variables if not (CACHE / f"{v}.npy").exists()]
    if not missing:
        return
    from clouddrift.datasets import gdp1h
    CACHE.mkdir(parents=True, exist_ok=True)
    ds = gdp1h()
    for v in missing:
        print(f"Downloading gdp1h {v} ...")
        np.save(CACHE / f"{v}.npy", ds[v].values)


def build_extract(chunk=2_000_000):
    """Keep valid SST readings in the study region and period, reading the global arrays in chunks."""
    raw = {v: np.load(CACHE / f"{v}.npy", mmap_mode="r") for v in ["lon", "lat", "time", "sst"]}
    ids = np.load(CACHE / "id.npy")
    last_row = np.cumsum(np.load(CACHE / "rowsize.npy"))
    parts = {k: [] for k in ["id", "time", "lon", "lat", "sst_degC"]}

    for a in range(0, len(raw["lon"]), chunk):
        b = a + chunk
        lon, lat = np.asarray(raw["lon"][a:b]), np.asarray(raw["lat"][a:b])
        in_R = ((lon >= R["lon"][0]) & (lon <= R["lon"][1])
                & (lat >= R["lat"][0]) & (lat <= R["lat"][1]))
        rows = np.flatnonzero(in_R)
        if len(rows) == 0:
            continue

        t = np.asarray(raw["time"][a:b])[rows]
        # Convert SST from Kelvin to Celsius
        sst_c = np.asarray(raw["sst"][a:b])[rows] - 273.15
        valid_sst = (t >= START) & (t < END) & np.isfinite(sst_c) & (sst_c >= -2) & (sst_c <= 35)
        rows = rows[valid_sst]

        parts["id"].append(ids[np.searchsorted(last_row, a + rows, side="right")])
        parts["time"].append(t[valid_sst])
        parts["lon"].append(lon[rows])
        parts["lat"].append(lat[rows])
        parts["sst_degC"].append(sst_c[valid_sst])

    np.savez(EXTRACT, **{k: np.concatenate(v) for k, v in parts.items()})


if not EXTRACT.exists():
    download_gdp1h(["lon", "lat", "time", "sst", "id", "rowsize"])
    build_extract()

obs = np.load(EXTRACT)
df = pd.DataFrame({k: obs[k] for k in ["id", "time", "lon", "lat", "sst_degC"]})

df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["doy"] = df["time"].dt.dayofyear

print(f"Observations with SST: {len(df):,}")

Observations with SST: 4,081,401
